# Feature selection benchmark: methods and combinations

Notebook non execute. Objectif: comparer plusieurs methodes de selection/suppression de features, seules et combinees, avec une evaluation equitable.

Evaluation:
- niveau global fold: precision / recall / F1 all-to-all avec seuil optimise sur train
- niveau table-pair: precision / recall / F1 groupe par `pair_id`
- comparaison par budget de features `k`
- stats de redondance restantes: max/median Spearman et nb de liens au-dessus de seuils

Important: ce notebook est genere sans execution.

In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import f1_score, precision_score, recall_score, precision_recall_curve, normalized_mutual_info_score
from sklearn.preprocessing import RobustScaler

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception:
    XGBClassifier = None
    HAS_XGBOOST = False

warnings.filterwarnings('ignore')

REPO = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
OUTPUT_ROOT = REPO / 'outputs' / 'exp_occidata'
FOLDS_ROOT = OUTPUT_ROOT / 'folds'
REPORT_DIR = OUTPUT_ROOT / 'reports' / 'meeting_baselines_vs_metamatch' / 'feature_selection_benchmark_methods_and_combinations'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

FAST_MODE = True
SEED = 42
DROP_OVERLAP = True
DROP_CONSTANT = True

TARGET_FEATURE_COUNTS = [5, 8, 10, 12, 15, 20]
CORR_THRESHOLDS = [0.50, 0.60, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]
NMI_THRESHOLDS = [0.15, 0.20, 0.25, 0.30, 0.40, 0.50]
VIF_THRESHOLDS = [5.0, 10.0, 20.0, 50.0]
MRMR_PENALTIES = [0.5, 1.0, 2.0]
REDUNDANCY_LEVELS = [0.50, 0.70, 0.85, 0.90]

if FAST_MODE:
    SAMPLE_PER_FOLD = 5_000
    EVAL_SAMPLE_PER_FOLD = 20_000
    N_ESTIMATORS = 80
else:
    SAMPLE_PER_FOLD = 0
    EVAL_SAMPLE_PER_FOLD = 0
    N_ESTIMATORS = 220


## 1. Load folds and features

In [2]:
def read_fold_table(fold_id, split):
    folder = FOLDS_ROOT / f'fold_{fold_id}'
    for suffix in ['parquet', 'csv.gz', 'csv']:
        path = folder / f'{split}.{suffix}'
        if path.exists():
            if suffix == 'parquet':
                return pd.read_parquet(path)
            return pd.read_csv(path)
    raise FileNotFoundError(f'Missing {split} file for fold {fold_id}')

def feature_family(feature):
    return feature.split('_', 1)[0]

def get_feature_columns(df):
    cols = [c for c in df.columns if c.startswith(('syn_', 'cls_', 'tda_'))]
    if DROP_OVERLAP:
        cols = [c for c in cols if 'overlap' not in c.lower()]
    if DROP_CONSTANT:
        nunique = df[cols].nunique(dropna=False)
        cols = [c for c in cols if nunique[c] > 1]
    return cols

def sample_train_fold(df, sample_n, seed):
    if sample_n == 0 or len(df) <= sample_n:
        return df.copy()
    pos = df[df['label'] == 1]
    neg = df[df['label'] == 0]
    n_pos = min(len(pos), max(1, sample_n // 3))
    n_neg = sample_n - n_pos
    pos_s = pos.sample(n=n_pos, random_state=seed, replace=len(pos) < n_pos)
    neg_s = neg.sample(n=min(n_neg, len(neg)), random_state=seed)
    return pd.concat([pos_s, neg_s], ignore_index=True).sample(frac=1.0, random_state=seed)

fold_ids = sorted(int(p.name.split('_')[-1]) for p in FOLDS_ROOT.glob('fold_*'))
fold0 = read_fold_table(fold_ids[0], 'train')
FEATURES = get_feature_columns(fold0)

sample_parts = []
for fid in fold_ids:
    train_df = read_fold_table(fid, 'train')
    sample_parts.append(sample_train_fold(train_df, SAMPLE_PER_FOLD, SEED + fid))

work_df = pd.concat(sample_parts, ignore_index=True)
X_raw = work_df[FEATURES].copy()
y = work_df['label'].astype(int).to_numpy()

display(pd.DataFrame({'feature': FEATURES, 'family': [feature_family(f) for f in FEATURES]}).groupby('family').size().rename('n_features').reset_index())
print('n_features after overlap removal:', len(FEATURES))


,family,n_features
0,cls,8
1,syn,19
2,tda,31


n_features after overlap removal: 58


## 2. Shared preprocessing and relevance

In [3]:
def make_numeric_matrix(X_df, scale='robust'):
    imputer = SimpleImputer(strategy='median')
    X_imp = imputer.fit_transform(X_df)
    if scale == 'robust':
        scaler = RobustScaler()
        X_scaled = scaler.fit_transform(X_imp)
    elif scale is None:
        scaler = None
        X_scaled = X_imp
    else:
        raise ValueError(scale)
    return X_scaled, imputer, scaler

X, _, _ = make_numeric_matrix(X_raw, scale='robust')
feature_names = np.array(FEATURES)

mi_target = mutual_info_classif(X, y, discrete_features=False, random_state=SEED, n_neighbors=3)
relevance_df = pd.DataFrame({
    'feature': feature_names,
    'family': [feature_family(f) for f in feature_names],
    'mi_target': mi_target,
}).sort_values('mi_target', ascending=False).reset_index(drop=True)

spearman_corr = pd.DataFrame(X, columns=feature_names).corr(method='spearman').fillna(0.0)
pearson_corr = pd.DataFrame(X, columns=feature_names).corr(method='pearson').fillna(0.0)
kendall_corr = pd.DataFrame(X, columns=feature_names).corr(method='kendall').fillna(0.0)

display(relevance_df.head(20))


,feature,family,mi_target
0,syn_jaro_winkler,syn,0.348813
1,syn_jaro,syn,0.347657
2,syn_cosine_bigrams,syn,0.345809
3,syn_cosine_trigrams,syn,0.330726
4,cls_euclidean,cls,0.323570
5,cls_cosine_dist,cls,0.323532
6,cls_cosine_sim,cls,0.323532
7,cls_pearson,cls,0.322977
8,cls_minkowski,cls,0.322902
9,cls_canberra,cls,0.322310


## 3. Selection methods

In [4]:
def top_k(features, k):
    return list(features)[:min(k, len(features))]

def order_by_relevance(features):
    rel = dict(zip(relevance_df['feature'], relevance_df['mi_target']))
    return sorted(list(features), key=lambda f: rel.get(f, 0.0), reverse=True)

def greedy_corr_prune(corr_df, threshold, pool=None):
    pool = list(FEATURES if pool is None else pool)
    remaining = set(pool)
    selected = []
    for feat in order_by_relevance(pool):
        if feat not in remaining:
            continue
        selected.append(feat)
        redundant = corr_df.index[corr_df[feat].abs() >= threshold].tolist()
        for r in redundant:
            remaining.discard(r)
    return selected

def discretize_for_mi(X_df, n_bins=10):
    out = pd.DataFrame(index=X_df.index)
    for col in X_df.columns:
        try:
            out[col] = pd.qcut(X_df[col].rank(method='first'), q=n_bins, labels=False, duplicates='drop')
        except Exception:
            out[col] = 0
    return out.fillna(0).astype(int)

def compute_nmi_matrix(X_df, n_bins=10):
    X_disc = discretize_for_mi(X_df, n_bins=n_bins)
    features = list(X_disc.columns)
    n = len(features)
    out = pd.DataFrame(np.eye(n), index=features, columns=features)
    for i in range(n):
        for j in range(i + 1, n):
            val = normalized_mutual_info_score(X_disc.iloc[:, i], X_disc.iloc[:, j])
            out.iloc[i, j] = val
            out.iloc[j, i] = val
    return out

nmi_matrix = compute_nmi_matrix(pd.DataFrame(X, columns=feature_names), n_bins=10)

def greedy_nmi_prune(threshold, pool=None):
    pool = list(FEATURES if pool is None else pool)
    remaining = set(pool)
    selected = []
    for feat in order_by_relevance(pool):
        if feat not in remaining:
            continue
        selected.append(feat)
        redundant = nmi_matrix.index[nmi_matrix[feat] >= threshold].tolist()
        for r in redundant:
            remaining.discard(r)
    return selected

def cluster_representatives(threshold, pool=None):
    pool = list(FEATURES if pool is None else pool)
    corr = spearman_corr.loc[pool, pool]
    dist = 1.0 - corr.abs().clip(0, 1)
    np.fill_diagonal(dist.values, 0.0)
    Z = linkage(squareform(dist.values, checks=False), method='average')
    labels = fcluster(Z, t=1.0 - threshold, criterion='distance')
    rel = dict(zip(relevance_df['feature'], relevance_df['mi_target']))
    selected = []
    for cluster_id in sorted(set(labels)):
        members = [f for f, lab in zip(pool, labels) if lab == cluster_id]
        selected.append(sorted(members, key=lambda f: rel.get(f, 0.0), reverse=True)[0])
    return order_by_relevance(selected)

def graph_independent_set(threshold, pool=None):
    pool = list(FEATURES if pool is None else pool)
    rel = dict(zip(relevance_df['feature'], relevance_df['mi_target']))
    selected = set(pool)
    corr = spearman_corr.loc[pool, pool]
    while True:
        edges = []
        current = list(selected)
        for i, a in enumerate(current):
            for b in current[i + 1:]:
                val = abs(corr.loc[a, b])
                if val >= threshold:
                    edges.append((a, b, val))
        if not edges:
            break
        conflict_count = {f: 0 for f in selected}
        conflict_strength = {f: 0.0 for f in selected}
        for a, b, val in edges:
            conflict_count[a] += 1
            conflict_count[b] += 1
            conflict_strength[a] += val
            conflict_strength[b] += val
        loser = sorted(selected, key=lambda f: (conflict_count[f], conflict_strength[f], -rel.get(f, 0.0)), reverse=True)[0]
        selected.remove(loser)
    return order_by_relevance(selected)

def vif_like_scores(X_df, features):
    X_arr, _, _ = make_numeric_matrix(X_df[features], scale='robust')
    rows = []
    for i, feat in enumerate(features):
        y_feat = X_arr[:, i]
        X_other = np.delete(X_arr, i, axis=1)
        if X_other.shape[1] == 0:
            r2 = 0.0
        else:
            model = Ridge(alpha=1.0, random_state=SEED)
            model.fit(X_other, y_feat)
            r2 = max(0.0, min(0.999999, model.score(X_other, y_feat)))
        rows.append({'feature': feat, 'vif_like': 1.0 / (1.0 - r2)})
    return pd.DataFrame(rows).sort_values('vif_like', ascending=False)

def iterative_vif_prune(threshold, pool=None, min_features=2):
    pool = list(FEATURES if pool is None else pool)
    current = list(pool)
    rel = dict(zip(relevance_df['feature'], relevance_df['mi_target']))
    while len(current) > min_features:
        scores = vif_like_scores(X_raw, current)
        max_row = scores.iloc[0]
        if max_row['vif_like'] <= threshold:
            break
        tied = scores[scores['vif_like'] >= max_row['vif_like'] * 0.999]['feature'].tolist()
        loser = sorted(tied, key=lambda f: rel.get(f, 0.0))[0]
        current.remove(loser)
    return order_by_relevance(current)

def mrmr_select(k, penalty=1.0, pool=None, redundancy_df=None):
    pool = list(FEATURES if pool is None else pool)
    redundancy_df = spearman_corr.abs() if redundancy_df is None else redundancy_df
    rel = dict(zip(relevance_df['feature'], relevance_df['mi_target']))
    candidates = set(pool)
    selected = []
    for _ in range(min(k, len(pool))):
        best = None
        best_score = -np.inf
        for feat in candidates:
            redundancy = 0.0 if not selected else float(np.mean([abs(redundancy_df.loc[feat, s]) for s in selected]))
            score = rel.get(feat, 0.0) - penalty * redundancy
            if score > best_score:
                best = feat
                best_score = score
        selected.append(best)
        candidates.remove(best)
    return selected

def family_aware_spearman(family_threshold=0.85, global_threshold=0.90, min_per_family=None):
    if min_per_family is None:
        min_per_family = {'syn': 2, 'cls': 2, 'tda': 2}
    selected = []
    for fam in ['syn', 'cls', 'tda']:
        fam_pool = [f for f in FEATURES if feature_family(f) == fam]
        fam_selected = greedy_corr_prune(spearman_corr, family_threshold, pool=fam_pool)
        if len(fam_selected) < min_per_family.get(fam, 0):
            extra = [f for f in order_by_relevance(fam_pool) if f not in fam_selected]
            fam_selected += extra[:max(0, min_per_family[fam] - len(fam_selected))]
        selected.extend(fam_selected)
    return greedy_corr_prune(spearman_corr, global_threshold, pool=selected)


## 4. Candidate sets: single methods and combinations

In [5]:
def make_result(method, hyperparams, selected):
    selected = list(dict.fromkeys(selected))
    selected_set = set(selected)
    removed = [f for f in FEATURES if f not in selected_set]
    fam_rows = []
    for fam in ['syn', 'cls', 'tda']:
        before = sum(feature_family(f) == fam for f in FEATURES)
        after = sum(feature_family(f) == fam for f in selected)
        fam_rows.append({'method': method, 'hyperparams': json.dumps(hyperparams, sort_keys=True), 'family': fam, 'before': before, 'after': after, 'removed': before - after})
    return {'method': method, 'hyperparams': hyperparams, 'selected': selected, 'removed': removed, 'n_selected': len(selected), 'n_removed': len(removed), 'family_stats': pd.DataFrame(fam_rows)}

all_results = []

for th in CORR_THRESHOLDS:
    all_results.append(make_result('greedy_spearman', {'threshold': th}, greedy_corr_prune(spearman_corr, th)))
    all_results.append(make_result('greedy_pearson', {'threshold': th}, greedy_corr_prune(pearson_corr, th)))
    all_results.append(make_result('greedy_kendall', {'threshold': th}, greedy_corr_prune(kendall_corr, th)))
    all_results.append(make_result('cluster_spearman', {'threshold': th}, cluster_representatives(th)))
    all_results.append(make_result('graph_independent_spearman', {'threshold': th}, graph_independent_set(th)))

for th in NMI_THRESHOLDS:
    all_results.append(make_result('greedy_nmi', {'threshold': th, 'n_bins': 10}, greedy_nmi_prune(th)))

for th in VIF_THRESHOLDS:
    all_results.append(make_result('iterative_vif_like', {'threshold': th}, iterative_vif_prune(th)))

for k in TARGET_FEATURE_COUNTS:
    for penalty in MRMR_PENALTIES:
        all_results.append(make_result('mrmr_spearman', {'k': k, 'penalty': penalty}, mrmr_select(k=k, penalty=penalty)))

for family_th in [0.80, 0.85, 0.90]:
    for global_th in [0.85, 0.90, 0.95]:
        all_results.append(make_result('family_aware_spearman', {'family_threshold': family_th, 'global_threshold': global_th}, family_aware_spearman(family_th, global_th)))

# Combination strategies.
for k in TARGET_FEATURE_COUNTS:
    mi_pool = relevance_df.head(max(2 * k, k + 5))['feature'].tolist()
    all_results.append(make_result('combo_mi_prefilter_graph', {'k': k, 'prefilter': len(mi_pool), 'graph_threshold': 0.85}, top_k(graph_independent_set(0.85, pool=mi_pool), k)))
    all_results.append(make_result('combo_mi_prefilter_mrmr', {'k': k, 'prefilter': len(mi_pool), 'penalty': 1.0}, mrmr_select(k=k, penalty=1.0, pool=mi_pool)))
    cluster_pool = cluster_representatives(0.85)
    all_results.append(make_result('combo_cluster_then_mrmr', {'k': k, 'cluster_threshold': 0.85, 'penalty': 1.0}, mrmr_select(k=k, penalty=1.0, pool=cluster_pool)))
    family_pool = family_aware_spearman(0.85, 0.95)
    all_results.append(make_result('combo_family_then_mrmr', {'k': k, 'family_threshold': 0.85, 'global_threshold': 0.95, 'penalty': 1.0}, mrmr_select(k=k, penalty=1.0, pool=family_pool)))
    mrmr_pool = mrmr_select(k=max(2 * k, k + 5), penalty=0.5)
    all_results.append(make_result('combo_mrmr_then_graph', {'k': k, 'mrmr_prefilter': len(mrmr_pool), 'graph_threshold': 0.85}, top_k(graph_independent_set(0.85, pool=mrmr_pool), k)))

candidate_summary = pd.DataFrame([{'method': r['method'], 'hyperparams': json.dumps(r['hyperparams'], sort_keys=True), 'n_selected': r['n_selected'], 'selected': ' | '.join(r['selected'])} for r in all_results])
display(candidate_summary.sort_values(['method', 'n_selected']))
candidate_summary.to_csv(REPORT_DIR / 'candidate_feature_sets.csv', index=False)


,method,hyperparams,n_selected,selected
3,cluster_spearman,"{""threshold"": 0.5}",16,syn_jaro_winkler | tda_h0_wasserstein | tda_h0...
8,cluster_spearman,"{""threshold"": 0.6}",21,syn_jaro_winkler | cls_euclidean | tda_h0_wass...
13,cluster_spearman,"{""threshold"": 0.7}",26,syn_jaro_winkler | syn_cosine_bigrams | cls_eu...
18,cluster_spearman,"{""threshold"": 0.75}",30,syn_jaro_winkler | syn_cosine_bigrams | cls_eu...
23,cluster_spearman,"{""threshold"": 0.8}",32,syn_jaro_winkler | syn_cosine_bigrams | cls_eu...
...,...,...,...,...
63,mrmr_spearman,"{""k"": 15, ""penalty"": 1.0}",15,syn_jaro_winkler | tda_h0_entropy_combined | t...
64,mrmr_spearman,"{""k"": 15, ""penalty"": 2.0}",15,syn_jaro_winkler | tda_h0_entropy_combined | s...
65,mrmr_spearman,"{""k"": 20, ""penalty"": 0.5}",20,syn_jaro_winkler | tda_h0_entropy_combined | c...
66,mrmr_spearman,"{""k"": 20, ""penalty"": 1.0}",20,syn_jaro_winkler | tda_h0_entropy_combined | t...


## 5. Redundancy stats

In [6]:
def redundancy_stats(features):
    if len(features) < 2:
        return {'max_abs_spearman': 0.0, 'median_abs_spearman': 0.0, **{f'n_pairs_abs_spearman_ge_{str(t).replace(".", "_")}': 0 for t in REDUNDANCY_LEVELS}}
    vals = spearman_corr.loc[features, features].abs().to_numpy()
    vals = vals[np.triu_indices_from(vals, k=1)]
    out = {'max_abs_spearman': float(vals.max()), 'median_abs_spearman': float(np.median(vals))}
    for t in REDUNDANCY_LEVELS:
        out[f'n_pairs_abs_spearman_ge_{str(t).replace(".", "_")}'] = int((vals >= t).sum())
    return out


## 6. Fold and pair_id evaluation

In [7]:
def best_threshold(y_true, y_score):
    precision, recall, thresholds = precision_recall_curve(y_true, y_score)
    if len(thresholds) == 0:
        return 0.5
    p = precision[1:]
    r = recall[1:]
    f1 = np.divide(2 * p * r, p + r, out=np.zeros_like(p), where=(p + r) > 0)
    return float(thresholds[int(np.nanargmax(f1))])

def build_eval_model(y_train, seed):
    if HAS_XGBOOST:
        n_pos = max(1, int(y_train.sum()))
        n_neg = max(1, int(len(y_train) - y_train.sum()))
        return XGBClassifier(n_estimators=N_ESTIMATORS, max_depth=4, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9, objective='binary:logistic', eval_metric='logloss', tree_method='hist', random_state=seed, n_jobs=1, scale_pos_weight=n_neg / n_pos)
    return LogisticRegression(class_weight='balanced', max_iter=1000, random_state=seed)

def evaluate_features(method, hyperparams, features):
    fold_rows = []
    pair_rows = []
    for fid in fold_ids:
        train_df = read_fold_table(fid, 'train')
        test_df = read_fold_table(fid, 'test')
        if 'pair_id' not in test_df.columns:
            raise KeyError('pair_id column is required for table-pair evaluation')
        if EVAL_SAMPLE_PER_FOLD and len(train_df) > EVAL_SAMPLE_PER_FOLD:
            train_df = sample_train_fold(train_df, EVAL_SAMPLE_PER_FOLD, SEED + fid)
        X_train, imp, sc = make_numeric_matrix(train_df[features], scale='robust')
        X_test = sc.transform(imp.transform(test_df[features])) if sc is not None else imp.transform(test_df[features])
        y_train = train_df['label'].astype(int).to_numpy()
        y_test = test_df['label'].astype(int).to_numpy()
        model = build_eval_model(y_train, SEED + fid)
        model.fit(X_train, y_train)
        train_score = model.predict_proba(X_train)[:, 1]
        test_score = model.predict_proba(X_test)[:, 1]
        thr = best_threshold(y_train, train_score)
        pred = (test_score >= thr).astype(int)
        fold_rows.append({'method': method, 'hyperparams': json.dumps(hyperparams, sort_keys=True), 'fold_id': fid, 'n_features': len(features), 'threshold_opt_train': thr, 'precision_all2all_opt_train': float(precision_score(y_test, pred, zero_division=0)), 'recall_all2all_opt_train': float(recall_score(y_test, pred, zero_division=0)), 'f1_all2all_opt_train': float(f1_score(y_test, pred, zero_division=0)), 'features': ' | '.join(features)})
        tmp = test_df[['pair_id', 'label']].copy()
        tmp['pred'] = pred
        for pair_id, part in tmp.groupby('pair_id'):
            pair_rows.append({'method': method, 'hyperparams': json.dumps(hyperparams, sort_keys=True), 'fold_id': fid, 'pair_id': pair_id, 'n_rows': len(part), 'n_pos': int(part['label'].sum()), 'precision_pair_opt_train': float(precision_score(part['label'], part['pred'], zero_division=0)), 'recall_pair_opt_train': float(recall_score(part['label'], part['pred'], zero_division=0)), 'f1_pair_opt_train': float(f1_score(part['label'], part['pred'], zero_division=0))})
    return pd.DataFrame(fold_rows), pd.DataFrame(pair_rows)


## 7. Run benchmark and summarize

In [8]:
summary_rows = []
family_rows = []
fold_evals = []
pair_evals = []

for res in all_results:
    fold_eval, pair_eval = evaluate_features(res['method'], res['hyperparams'], res['selected'])
    fold_evals.append(fold_eval)
    pair_evals.append(pair_eval)
    red = redundancy_stats(res['selected'])
    summary_rows.append({
        'method': res['method'],
        'hyperparams': json.dumps(res['hyperparams'], sort_keys=True),
        'n_selected': res['n_selected'],
        'n_removed': res['n_removed'],
        'mean_precision_folds': fold_eval['precision_all2all_opt_train'].mean(),
        'std_precision_folds': fold_eval['precision_all2all_opt_train'].std(),
        'mean_recall_folds': fold_eval['recall_all2all_opt_train'].mean(),
        'std_recall_folds': fold_eval['recall_all2all_opt_train'].std(),
        'mean_f1_folds': fold_eval['f1_all2all_opt_train'].mean(),
        'std_f1_folds': fold_eval['f1_all2all_opt_train'].std(),
        'mean_precision_pairs': pair_eval['precision_pair_opt_train'].mean(),
        'std_precision_pairs': pair_eval['precision_pair_opt_train'].std(),
        'mean_recall_pairs': pair_eval['recall_pair_opt_train'].mean(),
        'std_recall_pairs': pair_eval['recall_pair_opt_train'].std(),
        'mean_f1_pairs': pair_eval['f1_pair_opt_train'].mean(),
        'std_f1_pairs': pair_eval['f1_pair_opt_train'].std(),
        'n_pair_ids_eval': pair_eval['pair_id'].nunique(),
        'selected': ' | '.join(res['selected']),
        **red,
    })
    family_rows.append(res['family_stats'])

benchmark_summary = pd.DataFrame(summary_rows).sort_values(['mean_f1_pairs', 'mean_f1_folds'], ascending=[False, False])
family_summary = pd.concat(family_rows, ignore_index=True)
fold_evaluation = pd.concat(fold_evals, ignore_index=True)
pair_evaluation = pd.concat(pair_evals, ignore_index=True)

display(benchmark_summary)
display(family_summary)
display(fold_evaluation)
display(pair_evaluation)

benchmark_summary.to_csv(REPORT_DIR / 'benchmark_summary.csv', index=False)
family_summary.to_csv(REPORT_DIR / 'benchmark_family_summary.csv', index=False)
fold_evaluation.to_csv(REPORT_DIR / 'benchmark_fold_evaluation.csv', index=False)
pair_evaluation.to_csv(REPORT_DIR / 'benchmark_pair_id_evaluation.csv', index=False)


,method,hyperparams,n_selected,n_removed,mean_precision_folds,std_precision_folds,mean_recall_folds,std_recall_folds,mean_f1_folds,std_f1_folds,...,mean_f1_pairs,std_f1_pairs,n_pair_ids_eval,selected,max_abs_spearman,median_abs_spearman,n_pairs_abs_spearman_ge_0_5,n_pairs_abs_spearman_ge_0_7,n_pairs_abs_spearman_ge_0_85,n_pairs_abs_spearman_ge_0_9
12,greedy_kendall,"{""threshold"": 0.7}",34,24,0.646753,0.080032,0.676791,0.078658,0.653795,0.018679,...,0.576063,0.262129,551,syn_jaro_winkler | syn_cosine_bigrams | syn_co...,0.887570,0.119012,56,11,1,0
49,iterative_vif_like,"{""threshold"": 50.0}",47,11,0.650473,0.092143,0.672977,0.083757,0.651835,0.023656,...,0.573698,0.255886,551,syn_jaro_winkler | syn_jaro | syn_cosine_bigra...,0.999622,0.119345,142,48,21,14
72,family_aware_spearman,"{""family_threshold"": 0.85, ""global_threshold"":...",35,23,0.663693,0.076370,0.656285,0.076496,0.652707,0.011357,...,0.571990,0.259315,551,syn_jaro_winkler | syn_cosine_bigrams | syn_co...,0.887570,0.120485,63,12,2,0
73,family_aware_spearman,"{""family_threshold"": 0.85, ""global_threshold"":...",35,23,0.663693,0.076370,0.656285,0.076496,0.652707,0.011357,...,0.571990,0.259315,551,syn_jaro_winkler | syn_cosine_bigrams | syn_co...,0.887570,0.120485,63,12,2,0
17,greedy_kendall,"{""threshold"": 0.75}",36,22,0.657019,0.076337,0.657769,0.076798,0.650038,0.012200,...,0.571530,0.259329,551,syn_jaro_winkler | syn_cosine_bigrams | syn_co...,0.887570,0.127247,72,13,3,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55,mrmr_spearman,"{""k"": 8, ""penalty"": 2.0}",8,50,0.439928,0.110116,0.626170,0.107644,0.498962,0.062569,...,0.498311,0.275712,551,syn_jaro_winkler | tda_h0_entropy_combined | s...,0.449976,0.053628,0,0,0,0
54,mrmr_spearman,"{""k"": 8, ""penalty"": 1.0}",8,50,0.528791,0.122743,0.612135,0.104450,0.548913,0.043012,...,0.497997,0.269146,551,syn_jaro_winkler | tda_h0_entropy_combined | t...,0.663321,0.128558,4,0,0,0
50,mrmr_spearman,"{""k"": 5, ""penalty"": 0.5}",5,53,0.417202,0.086607,0.646438,0.085698,0.496013,0.047145,...,0.496515,0.273011,551,syn_jaro_winkler | tda_h0_entropy_combined | c...,0.449976,0.109675,0,0,0,0
78,combo_mi_prefilter_mrmr,"{""k"": 5, ""penalty"": 1.0, ""prefilter"": 10}",5,53,0.473081,0.089007,0.663438,0.080714,0.541834,0.043046,...,0.494980,0.264373,551,syn_jaro_winkler | cls_canberra | syn_cosine_t...,0.902799,0.597294,9,3,1,1


,method,hyperparams,family,before,after,removed
0,greedy_spearman,"{""threshold"": 0.5}",syn,19,3,16
1,greedy_spearman,"{""threshold"": 0.5}",cls,8,1,7
2,greedy_spearman,"{""threshold"": 0.5}",tda,31,10,21
3,greedy_pearson,"{""threshold"": 0.5}",syn,19,6,13
4,greedy_pearson,"{""threshold"": 0.5}",cls,8,0,8
...,...,...,...,...,...,...
316,combo_family_then_mrmr,"{""family_threshold"": 0.85, ""global_threshold"":...",cls,8,2,6
317,combo_family_then_mrmr,"{""family_threshold"": 0.85, ""global_threshold"":...",tda,31,11,20
318,combo_mrmr_then_graph,"{""graph_threshold"": 0.85, ""k"": 20, ""mrmr_prefi...",syn,19,8,11
319,combo_mrmr_then_graph,"{""graph_threshold"": 0.85, ""k"": 20, ""mrmr_prefi...",cls,8,2,6


,method,hyperparams,fold_id,n_features,threshold_opt_train,precision_all2all_opt_train,recall_all2all_opt_train,f1_all2all_opt_train,features
0,greedy_spearman,"{""threshold"": 0.5}",0,14,0.846671,0.509063,0.625696,0.561386,syn_jaro_winkler | cls_chebyshev | tda_h0_wass...
1,greedy_spearman,"{""threshold"": 0.5}",1,14,0.844084,0.487585,0.639526,0.553314,syn_jaro_winkler | cls_chebyshev | tda_h0_wass...
2,greedy_spearman,"{""threshold"": 0.5}",2,14,0.862239,0.517788,0.598952,0.555421,syn_jaro_winkler | cls_chebyshev | tda_h0_wass...
3,greedy_spearman,"{""threshold"": 0.5}",3,14,0.856686,0.496857,0.625765,0.553910,syn_jaro_winkler | cls_chebyshev | tda_h0_wass...
4,greedy_spearman,"{""threshold"": 0.5}",4,14,0.849617,0.522880,0.636541,0.574139,syn_jaro_winkler | cls_chebyshev | tda_h0_wass...
...,...,...,...,...,...,...,...,...,...
637,combo_mrmr_then_graph,"{""graph_threshold"": 0.85, ""k"": 20, ""mrmr_prefi...",1,20,0.867175,0.612094,0.614360,0.613225,syn_jaro_winkler | syn_cosine_bigrams | syn_co...
638,combo_mrmr_then_graph,"{""graph_threshold"": 0.85, ""k"": 20, ""mrmr_prefi...",2,20,0.893234,0.692656,0.571642,0.626358,syn_jaro_winkler | syn_cosine_bigrams | syn_co...
639,combo_mrmr_then_graph,"{""graph_threshold"": 0.85, ""k"": 20, ""mrmr_prefi...",3,20,0.870253,0.592128,0.627924,0.609501,syn_jaro_winkler | syn_cosine_bigrams | syn_co...
640,combo_mrmr_then_graph,"{""graph_threshold"": 0.85, ""k"": 20, ""mrmr_prefi...",4,20,0.821278,0.530026,0.711726,0.607582,syn_jaro_winkler | syn_cosine_bigrams | syn_co...


,method,hyperparams,fold_id,pair_id,n_rows,n_pos,precision_pair_opt_train,recall_pair_opt_train,f1_pair_opt_train
0,greedy_spearman,"{""threshold"": 0.5}",0,ChEMBL__Joinable__assays_both_50_1_ac2_ev,144,1,0.000000,0.000000,0.000000
1,greedy_spearman,"{""threshold"": 0.5}",0,ChEMBL__Joinable__assays_both_50_1_ac3_ev,144,1,0.000000,0.000000,0.000000
2,greedy_spearman,"{""threshold"": 0.5}",0,ChEMBL__Joinable__assays_both_50_30_ec_ev,210,6,0.500000,1.000000,0.666667
3,greedy_spearman,"{""threshold"": 0.5}",0,ChEMBL__Joinable__assays_both_50_70_ac3_ev,380,16,0.714286,0.625000,0.666667
4,greedy_spearman,"{""threshold"": 0.5}",0,ChEMBL__Joinable__assays_both_50_70_ac4_ev,380,16,0.204082,0.625000,0.307692
...,...,...,...,...,...,...,...,...,...
99933,combo_mrmr_then_graph,"{""graph_threshold"": 0.85, ""k"": 20, ""mrmr_prefi...",5,TPC-DI__View-Unionable__prospect_both_0_50_ac5_av,272,11,0.500000,0.636364,0.560000
99934,combo_mrmr_then_graph,"{""graph_threshold"": 0.85, ""k"": 20, ""mrmr_prefi...",5,TPC-DI__View-Unionable__prospect_both_0_50_ac5_ev,272,11,0.500000,0.636364,0.560000
99935,combo_mrmr_then_graph,"{""graph_threshold"": 0.85, ""k"": 20, ""mrmr_prefi...",5,TPC-DI__View-Unionable__prospect_both_0_70_ac1_ev,342,15,0.666667,0.666667,0.666667
99936,combo_mrmr_then_graph,"{""graph_threshold"": 0.85, ""k"": 20, ""mrmr_prefi...",5,TPC-DI__View-Unionable__prospect_both_0_70_ec_ev,342,15,0.571429,0.800000,0.666667


## 8. Fair comparison by selected feature budget

In [9]:
fair_rows = []
for target_k in TARGET_FEATURE_COUNTS:
    for method, part in benchmark_summary.groupby('method'):
        candidates = part.copy()
        candidates['target_k'] = target_k
        candidates['k_distance'] = (candidates['n_selected'] - target_k).abs()
        chosen = candidates.sort_values(['k_distance', 'mean_f1_pairs', 'mean_f1_folds', 'max_abs_spearman'], ascending=[True, False, False, True]).head(1)
        fair_rows.append(chosen)

fair_comparison = pd.concat(fair_rows, ignore_index=True).sort_values(['target_k', 'mean_f1_pairs', 'mean_f1_folds'], ascending=[True, False, False])
display(fair_comparison)
fair_comparison.to_csv(REPORT_DIR / 'fair_comparison_by_k.csv', index=False)


,method,hyperparams,n_selected,n_removed,mean_precision_folds,std_precision_folds,mean_recall_folds,std_recall_folds,mean_f1_folds,std_f1_folds,...,n_pair_ids_eval,selected,max_abs_spearman,median_abs_spearman,n_pairs_abs_spearman_ge_0_5,n_pairs_abs_spearman_ge_0_7,n_pairs_abs_spearman_ge_0_85,n_pairs_abs_spearman_ge_0_9,target_k,k_distance
12,iterative_vif_like,"{""threshold"": 5.0}",33,25,0.650995,0.086942,0.641359,0.079051,0.637935,0.022001,...,551,syn_jaro_winkler | syn_cosine_trigrams | cls_c...,0.999622,0.104473,41,7,4,2,5,28
6,family_aware_spearman,"{""family_threshold"": 0.8, ""global_threshold"": ...",32,26,0.620635,0.101438,0.666981,0.090183,0.631545,0.028315,...,551,syn_jaro_winkler | syn_cosine_bigrams | cls_eu...,0.769785,0.117331,44,9,0,0,5,27
8,greedy_kendall,"{""threshold"": 0.5}",22,36,0.626055,0.097104,0.640661,0.081990,0.622891,0.026579,...,551,syn_jaro_winkler | syn_cosine_trigrams | cls_e...,0.676980,0.119672,17,0,0,0,5,17
7,graph_independent_spearman,"{""threshold"": 0.5}",18,40,0.598697,0.092068,0.630947,0.088586,0.603543,0.023456,...,551,syn_cosine_bigrams | cls_canberra | tda_h0_bot...,0.495841,0.089233,0,0,0,0,5,13
9,greedy_nmi,"{""n_bins"": 10, ""threshold"": 0.15}",12,46,0.532265,0.097612,0.663833,0.107550,0.576768,0.035011,...,551,syn_jaro_winkler | cls_euclidean | tda_h0_wass...,0.621912,0.146421,6,0,0,0,5,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81,greedy_spearman,"{""threshold"": 0.6}",21,37,0.520744,0.060499,0.669842,0.076971,0.579829,0.018793,...,551,syn_jaro_winkler | cls_canberra | tda_h0_wasse...,0.597311,0.080113,10,0,0,0,20,1
77,graph_independent_spearman,"{""threshold"": 0.5}",18,40,0.598697,0.092068,0.630947,0.088586,0.603543,0.023456,...,551,syn_cosine_bigrams | cls_canberra | tda_h0_bot...,0.495841,0.089233,0,0,0,0,20,2
74,combo_mi_prefilter_mrmr,"{""k"": 20, ""penalty"": 1.0, ""prefilter"": 40}",20,38,0.560358,0.078029,0.660523,0.094023,0.596936,0.011650,...,551,syn_jaro_winkler | tda_h0_entropy_combined | t...,0.974663,0.168457,45,15,5,5,20,0
83,mrmr_spearman,"{""k"": 20, ""penalty"": 2.0}",20,38,0.558538,0.098975,0.650084,0.084978,0.589430,0.024465,...,551,syn_jaro_winkler | tda_h0_entropy_combined | s...,0.955011,0.071351,12,4,1,1,20,0


## 9. Final shortlist

Shortlist automatique:
- meilleur F1 pair_id par budget `k`
- meilleur compromis F1 / compacite
- meilleur compromis F1 / faible redondance

In [10]:
best_by_k = fair_comparison.sort_values(['target_k', 'mean_f1_pairs', 'mean_f1_folds'], ascending=[True, False, False]).groupby('target_k', as_index=False).head(1)
compact_best = benchmark_summary.assign(compact_score=benchmark_summary['mean_f1_pairs'] / np.sqrt(benchmark_summary['n_selected'].clip(lower=1))).sort_values('compact_score', ascending=False).head(10)
low_redundancy_best = benchmark_summary.assign(redundancy_penalized_score=benchmark_summary['mean_f1_pairs'] - 0.05 * benchmark_summary['max_abs_spearman']).sort_values('redundancy_penalized_score', ascending=False).head(10)

display(best_by_k)
display(compact_best)
display(low_redundancy_best)

final_shortlist = {
    'best_by_k': best_by_k.to_dict('records'),
    'compact_best': compact_best.to_dict('records'),
    'low_redundancy_best': low_redundancy_best.to_dict('records'),
}
with open(REPORT_DIR / 'final_shortlist.json', 'w') as f:
    json.dump(final_shortlist, f, indent=2)


,method,hyperparams,n_selected,n_removed,mean_precision_folds,std_precision_folds,mean_recall_folds,std_recall_folds,mean_f1_folds,std_f1_folds,...,n_pair_ids_eval,selected,max_abs_spearman,median_abs_spearman,n_pairs_abs_spearman_ge_0_5,n_pairs_abs_spearman_ge_0_7,n_pairs_abs_spearman_ge_0_85,n_pairs_abs_spearman_ge_0_9,target_k,k_distance
12,iterative_vif_like,"{""threshold"": 5.0}",33,25,0.650995,0.086942,0.641359,0.079051,0.637935,0.022001,...,551,syn_jaro_winkler | syn_cosine_trigrams | cls_c...,0.999622,0.104473,41,7,4,2,5,28
26,iterative_vif_like,"{""threshold"": 5.0}",33,25,0.650995,0.086942,0.641359,0.079051,0.637935,0.022001,...,551,syn_jaro_winkler | syn_cosine_trigrams | cls_c...,0.999622,0.104473,41,7,4,2,8,25
40,iterative_vif_like,"{""threshold"": 5.0}",33,25,0.650995,0.086942,0.641359,0.079051,0.637935,0.022001,...,551,syn_jaro_winkler | syn_cosine_trigrams | cls_c...,0.999622,0.104473,41,7,4,2,10,23
54,iterative_vif_like,"{""threshold"": 5.0}",33,25,0.650995,0.086942,0.641359,0.079051,0.637935,0.022001,...,551,syn_jaro_winkler | syn_cosine_trigrams | cls_c...,0.999622,0.104473,41,7,4,2,12,21
68,iterative_vif_like,"{""threshold"": 5.0}",33,25,0.650995,0.086942,0.641359,0.079051,0.637935,0.022001,...,551,syn_jaro_winkler | syn_cosine_trigrams | cls_c...,0.999622,0.104473,41,7,4,2,15,18
82,iterative_vif_like,"{""threshold"": 5.0}",33,25,0.650995,0.086942,0.641359,0.079051,0.637935,0.022001,...,551,syn_jaro_winkler | syn_cosine_trigrams | cls_c...,0.999622,0.104473,41,7,4,2,20,13


,method,hyperparams,n_selected,n_removed,mean_precision_folds,std_precision_folds,mean_recall_folds,std_recall_folds,mean_f1_folds,std_f1_folds,...,std_f1_pairs,n_pair_ids_eval,selected,max_abs_spearman,median_abs_spearman,n_pairs_abs_spearman_ge_0_5,n_pairs_abs_spearman_ge_0_7,n_pairs_abs_spearman_ge_0_85,n_pairs_abs_spearman_ge_0_9,compact_score
77,combo_mi_prefilter_graph,"{""graph_threshold"": 0.85, ""k"": 5, ""prefilter"":...",4,54,0.486283,0.110050,0.656917,0.090423,0.544058,0.054099,...,0.266031,551,syn_jaro_winkler | syn_cosine_bigrams | syn_co...,0.826969,0.635697,6,2,0,0,0.250462
81,combo_mrmr_then_graph,"{""graph_threshold"": 0.85, ""k"": 5, ""mrmr_prefil...",5,53,0.511102,0.100241,0.641727,0.092222,0.555540,0.042520,...,0.265741,551,syn_jaro_winkler | syn_cosine_bigrams | syn_co...,0.826969,0.558227,6,2,0,0,0.224882
79,combo_cluster_then_mrmr,"{""cluster_threshold"": 0.85, ""k"": 5, ""penalty"":...",5,53,0.445118,0.097228,0.624722,0.092984,0.506242,0.047772,...,0.279164,551,syn_jaro_winkler | tda_h0_entropy_combined | t...,0.449976,0.109675,0,0,0,0,0.224637
51,mrmr_spearman,"{""k"": 5, ""penalty"": 1.0}",5,53,0.445118,0.097228,0.624722,0.092984,0.506242,0.047772,...,0.279164,551,syn_jaro_winkler | tda_h0_entropy_combined | t...,0.449976,0.109675,0,0,0,0,0.224637
80,combo_family_then_mrmr,"{""family_threshold"": 0.85, ""global_threshold"":...",5,53,0.445118,0.097228,0.624722,0.092984,0.506242,0.047772,...,0.279164,551,syn_jaro_winkler | tda_h0_entropy_combined | t...,0.449976,0.109675,0,0,0,0,0.224637
50,mrmr_spearman,"{""k"": 5, ""penalty"": 0.5}",5,53,0.417202,0.086607,0.646438,0.085698,0.496013,0.047145,...,0.273011,551,syn_jaro_winkler | tda_h0_entropy_combined | c...,0.449976,0.109675,0,0,0,0,0.222048
78,combo_mi_prefilter_mrmr,"{""k"": 5, ""penalty"": 1.0, ""prefilter"": 10}",5,53,0.473081,0.089007,0.663438,0.080714,0.541834,0.043046,...,0.264373,551,syn_jaro_winkler | cls_canberra | syn_cosine_t...,0.902799,0.597294,9,3,1,1,0.221362
52,mrmr_spearman,"{""k"": 5, ""penalty"": 2.0}",5,53,0.407792,0.088349,0.634869,0.093922,0.484320,0.048525,...,0.277394,551,syn_jaro_winkler | tda_h0_entropy_combined | s...,0.449976,0.061284,0,0,0,0,0.219566
82,combo_mi_prefilter_graph,"{""graph_threshold"": 0.85, ""k"": 8, ""prefilter"":...",6,52,0.491534,0.093233,0.676199,0.082293,0.558654,0.031470,...,0.255979,551,syn_jaro_winkler | syn_cosine_bigrams | syn_co...,0.826969,0.556063,11,4,0,0,0.204678
53,mrmr_spearman,"{""k"": 8, ""penalty"": 0.5}",8,50,0.542555,0.091812,0.629928,0.080711,0.572395,0.027361,...,0.271262,551,syn_jaro_winkler | tda_h0_entropy_combined | c...,0.826969,0.160200,6,2,0,0,0.182879


,method,hyperparams,n_selected,n_removed,mean_precision_folds,std_precision_folds,mean_recall_folds,std_recall_folds,mean_f1_folds,std_f1_folds,...,std_f1_pairs,n_pair_ids_eval,selected,max_abs_spearman,median_abs_spearman,n_pairs_abs_spearman_ge_0_5,n_pairs_abs_spearman_ge_0_7,n_pairs_abs_spearman_ge_0_85,n_pairs_abs_spearman_ge_0_9,redundancy_penalized_score
12,greedy_kendall,"{""threshold"": 0.7}",34,24,0.646753,0.080032,0.676791,0.078658,0.653795,0.018679,...,0.262129,551,syn_jaro_winkler | syn_cosine_bigrams | syn_co...,0.887570,0.119012,56,11,1,0,0.531684
72,family_aware_spearman,"{""family_threshold"": 0.85, ""global_threshold"":...",35,23,0.663693,0.076370,0.656285,0.076496,0.652707,0.011357,...,0.259315,551,syn_jaro_winkler | syn_cosine_bigrams | syn_co...,0.887570,0.120485,63,12,2,0,0.527611
73,family_aware_spearman,"{""family_threshold"": 0.85, ""global_threshold"":...",35,23,0.663693,0.076370,0.656285,0.076496,0.652707,0.011357,...,0.259315,551,syn_jaro_winkler | syn_cosine_bigrams | syn_co...,0.887570,0.120485,63,12,2,0,0.527611
17,greedy_kendall,"{""threshold"": 0.75}",36,22,0.657019,0.076337,0.657769,0.076798,0.650038,0.012200,...,0.259329,551,syn_jaro_winkler | syn_cosine_bigrams | syn_co...,0.887570,0.127247,72,13,3,0,0.527151
19,graph_independent_spearman,"{""threshold"": 0.75}",32,26,0.608710,0.104717,0.670251,0.097809,0.625052,0.023972,...,0.261713,551,syn_jaro_winkler | syn_cosine_trigrams | cls_m...,0.743873,0.117414,46,7,0,0,0.524251
49,iterative_vif_like,"{""threshold"": 50.0}",47,11,0.650473,0.092143,0.672977,0.083757,0.651835,0.023656,...,0.255886,551,syn_jaro_winkler | syn_jaro | syn_cosine_bigra...,0.999622,0.119345,142,48,21,14,0.523717
74,family_aware_spearman,"{""family_threshold"": 0.9, ""global_threshold"": ...",33,25,0.610126,0.093623,0.681050,0.086654,0.633415,0.023488,...,0.263766,551,syn_jaro_winkler | syn_cosine_bigrams | syn_co...,0.826969,0.116399,49,10,0,0,0.522586
71,family_aware_spearman,"{""family_threshold"": 0.85, ""global_threshold"":...",33,25,0.610126,0.093623,0.681050,0.086654,0.633415,0.023488,...,0.263766,551,syn_jaro_winkler | syn_cosine_bigrams | syn_co...,0.826969,0.116399,49,10,0,0,0.522586
28,cluster_spearman,"{""threshold"": 0.85}",33,25,0.610126,0.093623,0.681050,0.086654,0.633415,0.023488,...,0.263766,551,syn_jaro_winkler | syn_cosine_bigrams | syn_co...,0.826969,0.116399,49,10,0,0,0.522586
25,greedy_spearman,"{""threshold"": 0.85}",33,25,0.610126,0.093623,0.681050,0.086654,0.633415,0.023488,...,0.263766,551,syn_jaro_winkler | syn_cosine_bigrams | syn_co...,0.826969,0.116399,49,10,0,0,0.522586
